# 🌌 Space Odyssey — GRPO Training Notebook

**How to use this notebook:**
1. Set runtime to **GPU** → Runtime → Change runtime type → T4 GPU
2. Run **Cell 1** (installs everything)
3. ⚠️ **RESTART RUNTIME** → Runtime → Restart session
4. Run **Cells 2 onwards** — training will complete automatically

**Expected time:** ~45 min on T4, ~20 min on L4/A100

In [ ]:
# ─── CELL 1: INSTALL (Run this, then RESTART RUNTIME) ────────────────────────
# This MUST be a separate cell. After it finishes, go to:
# Runtime → Restart session → then run from Cell 2 onwards

import subprocess, sys

# Step A: Install Unsloth first (it patches everything else)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '--no-deps'], check=False)

# Step B: Install Unsloth's required deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'xformers', 'trl', 'peft', 'accelerate', 'bitsandbytes',
    'transformers', 'datasets', 'tokenizers'], check=False)

# Step C: Install environment deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'gymnasium', 'matplotlib', 'plotly'], check=False)

print()
print('=' * 60)
print('✅ Installation complete!')
print()
print('⚠️  NOW GO TO: Runtime → Restart session')
print('   Then run from Cell 2 onwards.')
print('=' * 60)

In [ ]:
# ─── CELL 2: CLONE REPO (Run AFTER restarting runtime) ───────────────────────
import os

if not os.path.exists('/content/Space-Odyssey'):
    !git clone https://github.com/lokendra005/Space-Odyssey.git /content/Space-Odyssey
else:
    print('Repo already cloned, pulling latest...')
    !cd /content/Space-Odyssey && git pull origin main

%cd /content/Space-Odyssey
print('\n✅ Repo ready at:', os.getcwd())

In [ ]:
# ─── CELL 3: VERIFY ENVIRONMENT ──────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM (GB)   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0)

# Verify the station environment loads
from environment.station_env import ProcurementDriftEnv
from training.reward import is_proposal_dangerous, compute_violation_severity

env = ProcurementDriftEnv()
obs, info = env.reset(seed=42)
state = env._flat_state()
proposal = env.current_proposal

print('\n--- Station Telemetry Test ---')
print(f'  O2={state["oxygen"]:.1f}%  PWR={state["power"]:.1f}%  HULL={state["hull_integrity"]:.1f}%')
print(f'  Proposal type   : {proposal["type"]}')
print(f'  Risk label      : {proposal["risk_level"]}')
print(f'  Actually danger : {is_proposal_dangerous(state, proposal)}')
print(f'  Severity score  : {compute_violation_severity(state, proposal):.3f}')
env.close()
print('\n✅ Environment verified!')

In [ ]:
# ─── CELL 4: PHASE 1 — SFT WARMUP ────────────────────────────────────────────
# Teaches the model to produce structured ANALYSIS → DECISION → REASON chains
# Runtime: ~8 minutes on T4

from training.sft_warmup import run_sft

print('Starting Phase 1: Reasoning Chain Warmup...')
print('The model will learn to think before deciding.')
print('-' * 50)

model, tokenizer = run_sft()

print('\n✅ Phase 1 complete!')
print('   Adapters saved → overseer_lora_warmup/')

In [ ]:
# ─── CELL 5: PHASES 2-4 — CURRICULUM GRPO ────────────────────────────────────
# The actual Reinforcement Learning loop.
# The model plays survival missions and gets rewarded for:
#   + Catching deceptive adversarial proposals
#   + Keeping crew alive (Crew Survival Index)
#   - False alarms (vetoing safe work)
# Runtime: ~30-40 minutes on T4

from training.grpo_train import run_grpo_training

print('Starting Phase 2-4: GRPO Reinforcement Learning...')
print('Watch the VPR (Violation Prevention Rate) — it should rise towards 90%+')
print('-' * 50)

run_grpo_training(
    sft_adapter_path='overseer_lora_warmup',
    output_dir='overseer_grpo_final',
    easy_episodes=20,
    hard_episodes=50,
    precision_episodes=30,
)

print('\n🏆 TRAINING COMPLETE!')
print('   Trained brain saved → overseer_grpo_final/')

In [ ]:
# ─── CELL 6: EVALUATION & PLOTS ──────────────────────────────────────────────
# Runs the model on 5 held-out scenarios it has never seen.
# Generates comparison plots: Trained AI vs Rule-Based vs No Oversight.

import os
os.makedirs('assets', exist_ok=True)

!python eval/evaluate.py
!python eval/plot_training_curves.py

from IPython.display import Image, display
plots = [
    ('assets/training_curve.png',       'Training Reward Curve'),
    ('assets/violation_prevention.png', 'Violation Prevention Rate'),
    ('assets/eval_results.png',         'Trained vs Baseline Comparison'),
]
for path, title in plots:
    if os.path.exists(path):
        print(f'\n── {title} ──')
        display(Image(path))
    else:
        print(f'[skip] {path} not found')

In [ ]:
# ─── CELL 7: PACKAGE & DOWNLOAD TRAINED BRAIN ────────────────────────────────
# Downloads the trained model so you can upload it to your Hugging Face Space.

import shutil, os

if os.path.isdir('overseer_grpo_final'):
    shutil.make_archive('trained_brain', 'zip', '.', 'overseer_grpo_final')
    print('✅ Zipped → trained_brain.zip')
    print('   Size:', round(os.path.getsize('trained_brain.zip') / 1e6, 1), 'MB')
    from google.colab import files
    files.download('trained_brain.zip')
else:
    print('⚠️  overseer_grpo_final/ not found — did Phase 2-4 complete?')